In [33]:
import pandas as pd
from datetime import datetime
import numpy as np
import requests 
from io import StringIO
import json
from pathlib import Path


class PreparePastData:
    def __init__(self, patches_path: str = 'manual_odds_patches.json'):
       self.url =  "https://www.football-data.co.uk/mmz4281/{season}/{league}.csv"

       self.patches_path = Path(patches_path) 

       self.leagues = {
        'premier_league': 'E0',
        'bundesliga':     'D1',
        'la_liga':        'SP1',
        'serie_a':        'I1'}
       
       self.seasons = self._generate_seasons()
       self.raw = self._download_all()
       self.data = self._clean()
       self._apply_patches()

    def _generate_seasons(self) -> list[str]:
        current_year = datetime.now().year
        start_year = current_year - 10
        end_year = current_year + 1

        return [
            f"{str(y)[2:]}{str(y + 1)[2:]}"
            for y in range(start_year, end_year)
        ]

    def _download_season(self, league_code: str, season: str) -> pd.DataFrame | None:
        url = self.url.format(season=season, league=league_code)
        try:
            response = requests.get(url, timeout=10)
            if response.status_code == 200:
                df = pd.read_csv(StringIO(response.text), encoding='latin-1')
                df['season'] = season
                return df
        except requests.RequestException as e:
            print(f"Request error for {url}: {e}")
        return None
    
    def _download_all(self) -> pd.DataFrame:
        all_dfs = []
        for league_name, league_code in self.leagues.items():
            for season in self.seasons:
                df = self._download_season(league_code, season)
                if df is not None:
                    df['league'] = league_name
                    all_dfs.append(df)

        return pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame()
    
    def _clean(self) -> pd.DataFrame:
        COLS = ['Date', 'HomeTeam', 'AwayTeam', 'HS', 'AS', 'MaxH', 'MaxD', 'MaxA', 'league', 'season']

        df = self.raw[COLS].copy()
        df.columns = ['date', 'home_team', 'away_team', 'home_shots', 'away_shots',
                      'odds_h', 'odds_d', 'odds_a', 'league', 'season']

        df['date'] = pd.to_datetime(df['date'], format='%d/%m/%Y', errors='coerce')

        # Drop rows where match identity or shot data is missing
        before = len(df)
        df = df.dropna(subset=['home_team', 'away_team', 'home_shots', 'away_shots'])
        dropped = before - len(df)
        if dropped > 0:
            print(f"[PreparePastData] Dropped {dropped} rows with missing team or shot data.")

        # Convert odds to normalised probabilities
        df['prob_h'] = 1 / df['odds_h']
        df['prob_d'] = 1 / df['odds_d']
        df['prob_a'] = 1 / df['odds_a']

        total = df['prob_h'] + df['prob_d'] + df['prob_a']
        df['prob_h'] = df['prob_h'] / total
        df['prob_d'] = df['prob_d'] / total
        df['prob_a'] = df['prob_a'] / total

        # Flag rows with missing odds
        df['has_odds'] = df[['prob_h', 'prob_d', 'prob_a']].notna().all(axis=1)

        missing = df[~df['has_odds']][['date', 'home_team', 'away_team', 'league', 'season']]
        if not missing.empty:
            print(f"\n[PreparePastData] {len(missing)} rows are missing odds and will be excluded from model training:")
            print(missing.to_string(index=False))
            print("\nUse patch_odds() to manually supply odds for any of these matches.\n")

        return df
    

    def _apply_patches(self) -> None:
        if not self.patches_path.exists():
            return

        with open(self.patches_path, 'r') as f:
            patches = json.load(f)

        applied = 0
        for patch in patches:
            mask = (
                (self.data['home_team'] == patch['home_team']) &
                (self.data['away_team'] == patch['away_team']) &
                (self.data['date'] == pd.Timestamp(patch['date']))
            )
            if mask.any():
                self.data.loc[mask, 'odds_h'] = patch['odds_h']
                self.data.loc[mask, 'odds_d'] = patch['odds_d']
                self.data.loc[mask, 'odds_a'] = patch['odds_a']

                total = (1/patch['odds_h']) + (1/patch['odds_d']) + (1/patch['odds_a'])
                self.data.loc[mask, 'prob_h'] = (1/patch['odds_h']) / total
                self.data.loc[mask, 'prob_d'] = (1/patch['odds_d']) / total
                self.data.loc[mask, 'prob_a'] = (1/patch['odds_a']) / total
                self.data.loc[mask, 'has_odds'] = True
                applied += 1

        if applied > 0:
            print(f"[PreparePastData] Re-applied {applied} manual odds patch(es) from {self.patches_path}.")

    def patch_odds(self, home_team: str, away_team: str, date: str,
                   odds_h: float, odds_d: float, odds_a: float) -> None:

        patches = []
        if self.patches_path.exists():
            with open(self.patches_path, 'r') as f:
                patches = json.load(f)

        # Check if this match is already patched
        already_exists = any(
            p['home_team'] == home_team and
            p['away_team'] == away_team and
            p['date'] == date
            for p in patches
        )

        if already_exists:
            print(f"[PreparePastData] Patch for {home_team} vs {away_team} on {date} already exists. Skipping.")
            return

        patches.append({
            'home_team': home_team,
            'away_team': away_team,
            'date': date,
            'odds_h': odds_h,
            'odds_d': odds_d,
            'odds_a': odds_a
        })

        with open(self.patches_path, 'w') as f:
            json.dump(patches, f, indent=2)

        # Apply immediately to self.data in this session too
        self._apply_patches()
        print(f"[PreparePastData] Patch saved and applied for {home_team} vs {away_team} on {date}.")

In [34]:
pp = PreparePastData()

[PreparePastData] Dropped 1 rows with missing team or shot data.

[PreparePastData] 4339 rows are missing odds and will be excluded from model training:
      date          home_team          away_team         league season
       NaT            Burnley            Swansea premier_league   1617
       NaT     Crystal Palace          West Brom premier_league   1617
       NaT            Everton          Tottenham premier_league   1617
       NaT               Hull          Leicester premier_league   1617
       NaT           Man City         Sunderland premier_league   1617
       NaT      Middlesbrough              Stoke premier_league   1617
       NaT        Southampton            Watford premier_league   1617
       NaT            Arsenal          Liverpool premier_league   1617
       NaT        Bournemouth         Man United premier_league   1617
       NaT            Chelsea           West Ham premier_league   1617
       NaT         Man United        Southampton premier_league   

In [ ]:
# FIgure out if the code to get NA values is right or not

pp.data.loc(pp.data["odds_h"].isna)

ValueError: No axis named <bound method Series.isna of 0         NaN
1         NaN
2         NaN
3         NaN
4         NaN
         ... 
14261    3.00
14262    6.50
14263    3.40
14264    1.50
14265    3.55
Name: odds_h, Length: 14265, dtype: float64> for object type DataFrame